# 📘 Module 4.3 — DETR: Detection Transformer

**Goal:** Understand DETR — the first end-to-end Transformer-based object detector.

## Why DETR Matters for ADAS
DETR (DEtection TRansformer, Carion et al., 2020) eliminates the need for:
- Hand-designed anchor boxes
- Non-Maximum Suppression (NMS)
- Complex detection heads

It uses **set prediction** — directly predicting all objects in one shot.

```
Traditional:  Backbone → Anchors → NMS → Detections (many hand-crafted steps)
DETR:         Backbone → Transformer Encoder → Transformer Decoder → Detections (end-to-end!)
```

---

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 1. DETR Architecture

```
Image (3, H, W)
    │
    ▼ CNN Backbone (ResNet-50)
Feature Map (2048, H/32, W/32)
    │
    ▼ 1×1 Conv (reduce to d_model)
Reduced Features (256, H/32, W/32)
    │
    ▼ Flatten + Positional Encoding
Sequence (HW/1024, 256)
    │
    ▼ Transformer Encoder (6 layers)
Encoded Features
    │
    ▼ Transformer Decoder (6 layers)
    │  with N learnable "object queries"
    │
N Object Predictions:
  ├── Class (car, person, ..., no-object)
  └── Bounding Box (cx, cy, w, h)
```

In [ ]:
class SimpleDETR(nn.Module):
    """Simplified DETR for educational purposes."""
    
    def __init__(self, num_classes=91, num_queries=100, d_model=256, nhead=8,
                 num_encoder_layers=6, num_decoder_layers=6):
        super().__init__()
        
        # 1. CNN Backbone (simplified — real DETR uses ResNet-50)
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, d_model, 1),  # Reduce to d_model
        )
        
        # 2. Transformer
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=1024,
            batch_first=True,
        )
        
        # 3. Object Queries (learnable — each "slot" learns to detect a different object)
        self.query_embed = nn.Parameter(torch.randn(num_queries, d_model))
        
        # 4. Prediction Heads
        self.class_head = nn.Linear(d_model, num_classes + 1)  # +1 for "no object"
        self.bbox_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 4),  # (cx, cy, w, h) normalized
            nn.Sigmoid(),           # Bound to [0, 1]
        )
    
    def forward(self, x):
        batch_size = x.shape[0]
        
        # Extract features
        features = self.backbone(x)  # (B, d_model, H', W')
        
        # Flatten spatial dimensions to sequence
        B, C, H, W = features.shape
        src = features.flatten(2).transpose(1, 2)  # (B, H'*W', d_model)
        
        # Object queries
        queries = self.query_embed.unsqueeze(0).expand(batch_size, -1, -1)
        
        # Transformer: encoder processes image features, decoder attends to them
        decoder_out = self.transformer(src, queries)  # (B, num_queries, d_model)
        
        # Predict class and bbox for each query
        class_logits = self.class_head(decoder_out)  # (B, num_queries, num_classes+1)
        bbox_pred = self.bbox_head(decoder_out)      # (B, num_queries, 4)
        
        return class_logits, bbox_pred

# Test
detr = SimpleDETR(num_classes=5, num_queries=20, d_model=128,
                  nhead=8, num_encoder_layers=3, num_decoder_layers=3)

img = torch.randn(2, 3, 256, 256)
class_out, bbox_out = detr(img)

print(f"Input image: {img.shape}")
print(f"Class predictions: {class_out.shape}  (batch, queries, classes+1)")
print(f"Bbox predictions: {bbox_out.shape}   (batch, queries, 4)")
print(f"\nEach of {class_out.shape[1]} queries predicts one potential object")
print(f"Parameters: {sum(p.numel() for p in detr.parameters()):,}")

In [ ]:
# --- Hungarian Matching (how DETR assigns predictions to ground truth) ---
# During training, DETR uses the Hungarian algorithm to find the best
# one-to-one assignment between predictions and ground truth objects.

# This eliminates the need for NMS and anchor boxes!

print("DETR Training Process:")
print("1. Model outputs N predictions (e.g., 100)")
print("2. Ground truth has M objects (e.g., 5 cars + 2 pedestrians)")
print("3. Hungarian matching finds optimal 1-to-1 assignment")
print("4. Matched predictions: minimize class + bbox loss")
print("5. Unmatched predictions: should predict 'no object'")
print("")
print("✅ No anchors, no NMS — truly end-to-end!")

---
## ✅ Key Takeaways

1. **DETR** uses Transformers for end-to-end object detection — no anchors or NMS
2. **Object queries** are learnable embeddings that each predict one object
3. **Hungarian matching** provides the training supervision (optimal assignment)
4. DETR excels at **large objects** but struggles with small ones → solved by Deformable DETR
5. Modern variants (DINO, RT-DETR) are competitive with YOLO for real-time ADAS

---
## 📖 Next Steps
➡️ **Next module:** [05_LLMs/01_tokenization_embeddings.ipynb](../05_LLMs/01_tokenization_embeddings.ipynb) — Large Language Models